# Task B07: HMDA 2024 PySpark Cleaning Pipeline — `clean_all` and `analytic` (Rulebook v2.3)

**Project:** Decomposing Racial Gaps in Mortgage Denial Rates: A Large-Scale Analysis of the 2024 U.S. HMDA Market  
**Course:** Big Data Analysis (Final Project) — Group 3  
**Milestone:** Task B07 — Distributed PySpark implementation of rules T01–T12, V01–V10, X01–X11, F01–F03 (Plan steps B-5, B-7, B-8)  
**Companion documents:** `data_cleaning_plan.md`, `data_cleaning_rules.md` (v2.3), `decisions.md` (D-A … D-I), `Coding and documenting conventions.md`

### Purpose & Architectural Context
This notebook provides the **official Apache Spark (PySpark)** distributed implementation of the HMDA 2024 cleaning pipeline, serving as the direct counterpart to Task B06 (Pandas). 

> [!NOTE]
> **Key Technical Adaptations (Pandas vs PySpark):**
> 1. **Execution Model:** Replaced Pandas/PyArrow manual chunk streaming (`batch_size=250,000`) with a unified, distributed PySpark Catalyst Optimizer execution DAG.
> 2. **Three-Valued Logic (3VL):** Guarded all validity rules (V01–V08) with `F.coalesce(..., F.lit(False))` to ensure `NULL` values evaluate strictly to `False`, eliminating subtle filter/flag leakage across engines.
> 3. **High-Performance Mapping:** Replaced Python UDF serialization bottlenecks with native Catalyst expressions (`F.create_map()`) for DTI binning (T07).
> 4. **Row-wise Logic:** Replaced Pandas `axis=1` vectorization with compound boolean column expressions for Co-Applicant 4-state classification (X05) and exempt field aggregations (X09).
> 5. **Reproducibility:** Reuses exact Stage 0 constants (`B06_s4_winsor_thresholds.json`, `B06_lender_sizes.csv`, `B06_dup_table.csv`) to guarantee count-level parity against B05 and B06.

## Section 1: Configuration & PySpark Environment

- **Input:** `input/B04_raw_with_buckets.parquet`, `input/B06_s4_winsor_thresholds.json`, `input/B06_lender_sizes.csv`, `input/B06_dup_table.csv`.
- **Process:**
  - Configure Windows 8.3 short path for Python executable (`win32api.GetShortPathName`) to prevent subprocess spawning errors in Windows environments with spaces.
  - Configure Windows Hadoop NativeIO environment (`winutils.exe` & `hadoop.dll`) for local parquet file output without a Hadoop cluster.
  - Initialize local `SparkSession` with 10 GB driver memory, 8 shuffle partitions, and PyArrow optimization enabled.
  - Set paths using `pathlib.Path` declared once.
- **Output:** Active distributed `SparkSession` runtime environment.

> [!TIP]
> **[PYSPARK ADAPTATION NOTE: Windows Process Spawning, Short Paths & Hadoop NativeIO]**  
> 1. On Windows machines where user directories contain whitespace (e.g. `C:\Users\LE KHANH`), PySpark worker initialization defaults to searching for `python3` or fails on path unquoting. Resolving `GetShortPathName(sys.executable)` (`C:\Users\LEKHAN~1\...`) provides a clean path for `PYSPARK_PYTHON` and `PYSPARK_DRIVER_PYTHON`.
> 2. For local parquet output on Windows, Hadoop FileOutputCommitter requires `winutils.exe` and `hadoop.dll` under `HADOOP_HOME/bin` to prevent POSIX permission check failures.

## Section 1.1: Operating System & Hadoop NativeIO Setup

- **Input:** Local Windows OS runtime environment and Python interpreter.
- **Process:**
  - Resolve 8.3 short path via `win32api.GetShortPathName` to prevent whitespace subprocess errors (`C:\Users\LE KHANH`).
  - Configure `HADOOP_HOME` and system `PATH` with local `winutils.exe` and `hadoop.dll` for POSIX permission bypass.
- **Output:** Sanitized execution environment for PySpark worker subprocesses.

In [1]:
# Step 1: Windows environment, short path resolution & Hadoop NativeIO
import os, sys, json, time, shutil, platform
from pathlib import Path
from itertools import chain

try:
    import win32api
    SHORT_PYTHON = win32api.GetShortPathName(sys.executable)
except Exception:
    SHORT_PYTHON = sys.executable

os.environ["PYSPARK_PYTHON"] = SHORT_PYTHON
os.environ["PYSPARK_DRIVER_PYTHON"] = SHORT_PYTHON

SCRIPT_DIR = Path.cwd()
HADOOP_DIR = SCRIPT_DIR.parent / "hadoop"
if not HADOOP_DIR.exists():
    HADOOP_DIR = SCRIPT_DIR / "hadoop"
if HADOOP_DIR.exists():
    os.environ["HADOOP_HOME"] = str(HADOOP_DIR)
    os.environ["PATH"] = str(HADOOP_DIR / "bin") + ";" + os.environ.get("PATH", "")
print(f"Environment configured with Python short path: {SHORT_PYTHON}")

Environment configured with Python short path: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


## Section 1.2: PySpark Session & Storage Paths

- **Input:** Runtime environment and storage directories (`input/`, `output/`).
- **Process:**
  - Declare directory paths once using `pathlib.Path`.
  - Initialize `SparkSession` with 10 GB heap memory, 8 shuffle partitions, and PyArrow acceleration.
- **Output:** Active local `SparkSession` instance.

In [2]:
# Step 2: PySpark session & storage paths
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, ByteType, BooleanType
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# [CHANGE] Step 2: paths. Read B04/B06 outputs directly; a local input/ copy still wins if present.
CWD = Path.cwd()
ROOT = next((p for p in [CWD, *CWD.parents] if (p / "origin").exists()), CWD)
TASK_DIR = CWD if CWD.name.startswith("B07") else next((p for p in ROOT.glob("B07*") if p.is_dir()), CWD)
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_input(name, *fallback_dirs):
    candidates = [INPUT_DIR / name, *[ROOT / d / name for d in fallback_dirs]]
    path = next((p for p in candidates if p.exists()), None)
    assert path is not None, f"Missing {name}. Looked in: {[str(c) for c in candidates]}"
    return path

INPUT_PARQUET = find_input("B04_raw_with_buckets.parquet", "B04 - 30 Oct/output")
CLEAN_ALL_PATH = OUTPUT_DIR / "B07_clean_all.parquet"
ANALYTIC_PATH = OUTPUT_DIR / "B07_analytic.parquet"

spark_builder = (SparkSession.builder
    .master("local[*]")
    .appName("Task_B07_HMDA_PySpark_Cleaning")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true"))
if HADOOP_DIR.exists():
    spark_builder = spark_builder.config("spark.driver.extraLibraryPath", str(HADOOP_DIR / "bin"))
spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark {spark.version} ready | Heap: 10g | Shuffle partitions: 8")


PySpark 4.2.0 ready | Heap: 10g | Shuffle partitions: 8


## Section 1.1: Input Schema & Column Specifications

- **Input:** Raw Parquet metadata of `B04_raw_with_buckets.parquet`.
- **Process:**
  - Read input schema via PySpark; verify exactly 99 raw HMDA columns plus `row_id` and `bucket` (101 total).
  - Declare column groups matching B06: 23 Dictionary `Number` columns (T02 = 14 with `NA`/`Exempt` tokens + 9 others), 10 integer columns, 15 T06 exempt columns, 14 literal `Exempt` columns, and repeated fields (T09).
- **Output:** Column group constants and validation assertions.

> [!NOTE]
> **[PYSPARK ADAPTATION NOTE: Schema Declarations]**  
> Defining column groups as Python lists allows PySpark projection operations (`select`) to be applied programmatically without repetitive code.

In [3]:
# Step 1: Read raw input schema
raw_df = spark.read.parquet(str(INPUT_PARQUET))
ALL_COLS = raw_df.columns
RAW_COLS = [c for c in ALL_COLS if c not in ("row_id", "bucket")]
TOTAL_RAW_ROWS = 12_259_199
assert len(RAW_COLS) == 99 and "row_id" in ALL_COLS and "bucket" in ALL_COLS
print(f"Input: {INPUT_PARQUET.name} | Total columns: {len(ALL_COLS)} (99 raw HMDA + row_id + bucket)")

# Step 2: Numeric column groups (Rules T02 / T03)
T02_COLS = [
    "income", "property_value", "combined_loan_to_value_ratio", "interest_rate", "rate_spread",
    "total_loan_costs", "total_points_and_fees", "origination_charges", "discount_points",
    "lender_credits", "loan_term", "prepayment_penalty_term", "intro_rate_period",
    "multifamily_affordable_units"
]
NUM_OTHER = [
    "activity_year", "loan_amount", "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage", "tract_owner_occupied_units",
    "tract_one_to_four_family_homes", "tract_median_age_of_housing_units"
]
NUM_COLS = T02_COLS + NUM_OTHER
INT_COLS = [
    "activity_year", "loan_term", "prepayment_penalty_term", "intro_rate_period",
    "multifamily_affordable_units", "tract_population", "tract_owner_occupied_units",
    "tract_one_to_four_family_homes", "tract_median_age_of_housing_units",
    "ffiec_msa_md_median_family_income"
]
T12_TRACT6 = [
    "tract_population", "tract_minority_population_percent", "tract_owner_occupied_units",
    "tract_one_to_four_family_homes", "tract_median_age_of_housing_units",
    "tract_to_msa_income_percentage"
]
FFIEC_COL = "ffiec_msa_md_median_family_income"
assert len(NUM_COLS) == 23 and set(INT_COLS) <= set(NUM_COLS)

Input: B04_raw_with_buckets.parquet | Total columns: 101 (99 raw HMDA + row_id + bucket)


## Section 1.1b: Categorical Column Groups & Lookup Lists

- **Input:** HMDA 2024 Data Dictionary field lists.
- **Process:** Define column subsets for multi-race fields, age sentinels, token-cleaned features, and exempt groups.
- **Output:** Global tuples of categorical column identifiers.

In [4]:
# Step 1: Categorical column groups (Rules T05, T06, T09, X09)
DOC_1111_COLS = [
    "aus_1", "manufactured_home_land_property_interest", "balloon_payment", "negative_amortization",
    "other_nonamortizing_features", "interest_only_payment", "manufactured_home_secured_property_type",
    "reverse_mortgage", "applicant_credit_score_type", "co_applicant_credit_score_type",
    "submission_of_application", "initially_payable_to_institution", "business_or_commercial_purpose",
    "open_end_line_of_credit"
]
T06_WHITELIST = DOC_1111_COLS + ["denial_reason_1"]
LIT_EXEMPT_COLS = [c for c in T02_COLS if c != "income"] + ["debt_to_income_ratio"]
ID_SPECS = {"lei": 20, "state_code": 2, "county_code": 5, "census_tract": 11, "derived_msa_md": 5}
REPEATED_COLS = (
    [f"{p}_{k}" for p in ("applicant_ethnicity", "co_applicant_ethnicity", "applicant_race", "co_applicant_race") for k in range(2, 6)]
    + [f"aus_{k}" for k in range(2, 6)] + [f"denial_reason_{k}" for k in range(2, 5)]
)

# Step 2: Valid code sets for co-applicant states (X05) and ages (T08)
AGE_VALID = ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"]
SEX_PRES = ["1", "2", "3", "6"]
RACE_PRES = ["1", "2", "21", "22", "23", "24", "25", "26", "27", "3", "4", "41", "42", "43", "44", "5", "6"]
ETH_PRES = ["1", "11", "12", "13", "14", "2", "3"]

# DTI Mapping Lookups
DTI_POINTS = [str(i) for i in range(36, 50)]
DTI_FINE = {"<20": "<20", "20-<30": "20-<30", "30-<36": "30-<36", "50-60": "50-60", ">60": ">60"}
DTI_COARSE = dict(DTI_FINE)
for _i in range(36, 50):
    DTI_FINE[str(_i)] = str(_i)
    DTI_COARSE[str(_i)] = "36-40" if _i <= 40 else ("41-45" if _i <= 45 else "46-49")

def status_col_name(c):
    return "dti_status" if c == "debt_to_income_ratio" else f"{c}_status"
print(f"Schema specifications locked: {len(NUM_COLS)} numeric, {len(T06_WHITELIST)} exempt whitelist, {len(ID_SPECS)} ID fields")

Schema specifications locked: 23 numeric, 15 exempt whitelist, 5 ID fields


## Section 1.2: Golden Targets from B05 / B06

- **Input:** Empirical benchmarks established in B05 (`B05 - 31 Oct v5.ipynb`) and confirmed in B06 (`B06 - Clean.ipynb`).
- **Process:** Store the 45 target metrics in a dictionary for strict automated verification at the conclusion of the Spark pipeline.
- **Output:** `GOLDEN` dictionary.

In [5]:
# Step 1: Golden verification targets (B05 DuckDB audit / B06 Pandas pipeline)
GOLDEN = {
    "raw_rows": 12_259_199, "f01_rows": 8_661_779, "f02_rows": 8_170_177, "s0_rows": 8_136_664,
    "s0_denied": 2_003_276, "f03_alone_on_f01": 33_569, "f02_f03_overlap": 56,
    "f02_1111_in_s0": 250_691, "f03_1111_in_s0": 250_889,
    "v01_raw": 0, "v01_s0": 0, "v02_raw": 165_254, "v02_s0": 91_261, "v03_raw": 5_323, "v03_s0": 4_026,
    "v04_raw": 83_443, "v04_s0": 60_717, "v05_raw": 2_893, "v05_s0": 18,
    "v06_raw": 0, "v07_raw": 0, "v08_raw": 0,
    "v10_clusters": 20_229, "v10_extra_copies": 26_755,
    "x05_discordant_raw": 79_088, "x05_discordant_s0": 18_230, "x05_all_na_raw": 1_134_611, "x05_all_na_s0": 3_625,
    "x09_full_raw": 276_755, "x09_full_s0": 241_108, "x09_partial_raw": 23_164, "x09_partial_s0": 15_292,
    "dr1_is_1111_raw": 247_789,
    "t08_applicant_age_8888": 1_463_684, "t08_co_age_8888": 1_178_955, "t08_co_age_9999": 6_666_402,
    "t05_lowercase_na_tract": 1, "t05_non_msa_raw": 1_781_406,
    "tract_null_raw": 349_024, "tract_null_s0": 96_065,
    "ffiec_no_geo_raw": 293_681, "ffiec_no_geo_s0": 63_462, "median_age_na_code_raw": 204,
    "tract_income_zero_flag_raw": 58_174,
    "s5_all_rows": 8_041_498, "s5_all_denied": 1_944_293,
    "s5_keep_v02_rows": 8_132_620, "s5_keep_v02_denied": 1_999_367,
}
print(f"{len(GOLDEN)} golden verification targets loaded.")

48 golden verification targets loaded.


## Section 2: Reusing Stage 0 Constants (Plan B-5 / Decision D-F)

- **Input:** `B06_s4_winsor_thresholds.json`, `B06_lender_sizes.csv`, `B06_dup_table.csv`.
- **Process:**
  - Read S4 winsorization thresholds (exact linear quantiles calculated on S0).
  - Read lender sizes (application count per LEI) and duplicate cluster tables as Spark DataFrames.
- **Output:** Broadcast-ready DataFrames and threshold constants.

> [!IMPORTANT]
> **[PYSPARK ADAPTATION NOTE: Broadcast Hash Join for Stage 0 Lookups]**  
> In Pandas, Stage 0 used Python dictionaries with `Series.map()`. In PySpark, we leverage `F.broadcast(df)` for `lender_df_spark` and `dup_df_spark`. Because these tables are small (~100 KB - 700 KB), Spark distributes them to all executor nodes via Broadcast Hash Join ($O(1)$ memory lookup), avoiding an expensive network shuffle of the 12.26M rows.

In [6]:
# Step 1: Load S4 thresholds and Stage 0 tables
s4_json_path = find_input("B06_s4_winsor_thresholds.json", "B06 - Clean/output")
lender_csv_path = find_input("B06_lender_sizes.csv", "B06 - Clean/output")
dup_csv_path = find_input("B06_dup_table.csv", "B06 - Clean/output")

with open(s4_json_path, "r", encoding="utf-8") as f:
    S4_CONSTANTS = json.load(f)["thresholds"]

lender_df_spark = spark.read.csv(str(lender_csv_path), header=True, inferSchema=True)
dup_df_spark = spark.read.csv(str(dup_csv_path), header=True, inferSchema=True)

print(f"Stage 0 tables loaded: Lender counts ({lender_df_spark.count():,} LEIs) | Duplicates ({dup_df_spark.count():,} rows)")

Stage 0 tables loaded: Lender counts (4,926 LEIs) | Duplicates (46,984 rows)


## Section 3.1: Stage 1 — Trim and Raw-Token Status Columns (Rules T01, T02, T05)

- **Input:** Raw Spark DataFrame.
- **Process:**
  - **T01:** Trim whitespace across all 99 string columns.
  - Track lowercase `"na"` anomaly in `census_tract` before case conversions.
  - **T02 / T05:** Create `<col>_status` (`blank`, `na`, `exempt`, `value`). For `derived_msa_md`, code `'99999'` is assigned status `'non_msa'`.
- **Output:** Spark DataFrame with 19 status columns added.

> [!TIP]
> **[PYSPARK ADAPTATION NOTE: Single Projection Planning]**  
> Rather than invoking `df.withColumn()` in a 99-iteration loop (which bloats Spark's Catalyst AST plan), we build a list of projection expressions and apply them in a single `df.select()` call.

In [7]:
# Step 1: Projection trim for all 99 raw columns (T01)
trim_exprs = [F.trim(F.col(c)).alias(c) if c in RAW_COLS else F.col(c) for c in raw_df.columns]
df = raw_df.select(trim_exprs)

# Track single lowercase 'na' in census_tract
df = df.withColumn("flag_lowercase_na_tract", F.col("census_tract") == "na")

# Step 2: Token status expressions (T02, T05)
def make_token_status_expr(col_name):
    c = F.col(col_name)
    up = F.upper(c)
    return (
        F.when(c.isNull() | (c == ""), F.lit("blank"))
         .when(up == "NA", F.lit("na"))
         .when(up == "EXEMPT", F.lit("exempt"))
         .otherwise(F.lit("value"))
    )

for c in T02_COLS + list(ID_SPECS):
    if c == "derived_msa_md":
        df = df.withColumn(
            "derived_msa_md_status",
            F.when(F.col("derived_msa_md") == "99999", F.lit("non_msa"))
             .otherwise(make_token_status_expr("derived_msa_md"))
        )
    else:
        df = df.withColumn(f"{c}_status", make_token_status_expr(c))
print(f"T01 trim and T02/T05 status columns applied ({len(T02_COLS)+len(ID_SPECS)} status fields).")

T01 trim and T02/T05 status columns applied (19 status fields).


## Section 3.2: Stage 1 — DTI Mapping (T07) and Age Special Codes (T08)

- **Input:** DataFrame with raw tokens.
- **Process:**
  - **T07:** Strip `%` from `debt_to_income_ratio`. Map to `dti_bin_fine`, `dti_bin_coarse`, and numeric `dti_point` (36–49) using native Spark Map expressions.
  - **T08:** Map `8888` ("Not applicable") to NULL with status `'na'`. For co-applicant, map `9999` ("No co-applicant") to NULL with status `'no_coapp'`.
- **Output:** Cleaned DTI and age columns with corresponding status indicators.

> [!NOTE]
> **[PYSPARK ADAPTATION NOTE: Zero-UDF Vectorized Map Lookup]**  
> In Pandas, mapping dictionaries run via `Series.map()`. In PySpark, using a Python UDF degrades performance by forcing data serialization through the Python worker. We use native Spark `F.create_map()` with flattened literal items, allowing Spark to perform map lookups directly in native JVM bytecode.

In [8]:
# Step 1: Rule T07 DTI Mapping via native Catalyst map
fine_map_expr = F.create_map([F.lit(x) for x in chain(*DTI_FINE.items())])
coarse_map_expr = F.create_map([F.lit(x) for x in chain(*DTI_COARSE.items())])
dti_st_expr = make_token_status_expr("debt_to_income_ratio")
dti_clean_str = F.regexp_replace(F.col("debt_to_income_ratio"), "%", "")

df = df.withColumn("dti_status", dti_st_expr)
df = df.withColumn("debt_to_income_ratio", F.when(F.col("dti_status") == "value", dti_clean_str).otherwise(F.lit(None).cast(StringType())))
df = df.withColumn("dti_bin_fine", F.when(F.col("dti_status") == "value", fine_map_expr[F.col("debt_to_income_ratio")]).otherwise(F.lit(None).cast(StringType())))
df = df.withColumn("dti_bin_coarse", F.when(F.col("dti_status") == "value", coarse_map_expr[F.col("debt_to_income_ratio")]).otherwise(F.lit(None).cast(StringType())))
df = df.withColumn("dti_point", F.when((F.col("dti_status") == "value") & F.col("debt_to_income_ratio").isin(DTI_POINTS), F.col("debt_to_income_ratio").cast(DoubleType())).otherwise(F.lit(None).cast(DoubleType())))
print("Rule T07 DTI fine/coarse mapping applied successfully.")

Rule T07 DTI fine/coarse mapping applied successfully.


## Section 3.2b: Rule X05 Co-Applicant 4-State Classification

- **Input:** `co_applicant_age`, `co_applicant_sex`, `co_applicant_race_1`, `co_applicant_ethnicity_1`.
- **Process:** Classify co-applicant existence across 4 fields on raw values (BEFORE age sentinels are converted to NULL).
- **Output:** `has_coapplicant` indicator and `flag_coapp_discordant` consistency audit flag.

> [!IMPORTANT]
> **[EXECUTION ORDER DEPENDENCY]**  
> Rule X05 must execute BEFORE Rule T08 age sentinels nullification so that code `9999` (No Co-applicant) and `8888` (NA) in `co_applicant_age` are detected accurately rather than evaluated against `NULL`.

In [9]:
# Step 1: Rule X05 Co-Applicant State Mapping
def state4_expr(col_name, pres, no_c, na_t, blank_is_na=False):
    c = F.col(col_name)
    na_cond = c.isin(na_t)
    if blank_is_na:
        na_cond = na_cond | c.isNull() | (c == "")
    return (
        F.when(c.isin(pres), F.lit("Present"))
         .when(c.isin(no_c), F.lit("No_Coapp"))
         .when(na_cond, F.lit("NA"))
         .otherwise(F.lit("Invalid"))
    )

st_age = state4_expr("co_applicant_age", AGE_VALID, ["9999"], ["8888"], False)
st_sex = state4_expr("co_applicant_sex", SEX_PRES, ["5"], ["4"], False)
st_race = state4_expr("co_applicant_race_1", RACE_PRES, ["8"], ["7"], True)
st_eth = state4_expr("co_applicant_ethnicity_1", ETH_PRES, ["5"], ["4"], True)

# Step 2: Precedence Present > No_Coapp > NA
any_p = (st_age == "Present") | (st_sex == "Present") | (st_race == "Present") | (st_eth == "Present")
any_n = (st_age == "No_Coapp") | (st_sex == "No_Coapp") | (st_race == "No_Coapp") | (st_eth == "No_Coapp")

df = df.withColumn(
    "has_coapplicant",
    F.when(any_p, F.lit(1).cast(ByteType()))
     .when(any_n, F.lit(0).cast(ByteType()))
     .otherwise(F.lit(None).cast(ByteType()))
)
df = df.withColumn(
    "flag_coapp_discordant",
    (st_age != st_sex) | (st_age != st_race) | (st_age != st_eth)
)
print("X05 co-applicant 4-state precedence applied.")

X05 co-applicant 4-state precedence applied.


## Section 3.3: Rule T08 Applicant and Co-Applicant Age Sentinels

- **Input:** `applicant_age`, `co_applicant_age`.
- **Process:** Classify age bins and mask sentinels `8888` (`na`) and `9999` (`no_coapp`) to `NULL`.
- **Output:** `applicant_age_status`, `co_applicant_age_status` and sanitized age values.

In [10]:
# Step 2: Rule T08 Age Sentinels
app_age_st = (
    F.when(F.col("applicant_age").isNull() | (F.col("applicant_age") == ""), F.lit("blank"))
     .when(F.col("applicant_age").isin(AGE_VALID), F.lit("value"))
     .when(F.col("applicant_age") == "8888", F.lit("na"))
     .otherwise(F.lit("invalid"))
)
co_age_st = (
    F.when(F.col("co_applicant_age").isNull() | (F.col("co_applicant_age") == ""), F.lit("blank"))
     .when(F.col("co_applicant_age").isin(AGE_VALID), F.lit("value"))
     .when(F.col("co_applicant_age") == "8888", F.lit("na"))
     .when(F.col("co_applicant_age") == "9999", F.lit("no_coapp"))
     .otherwise(F.lit("invalid"))
)
df = df.withColumn("applicant_age_status", app_age_st)
df = df.withColumn("applicant_age", F.when(F.col("applicant_age_status") == "value", F.col("applicant_age")).otherwise(F.lit(None).cast(StringType())))
df = df.withColumn("co_applicant_age_status", co_age_st)
df = df.withColumn("co_applicant_age", F.when(F.col("co_applicant_age_status") == "value", F.col("co_applicant_age")).otherwise(F.lit(None).cast(StringType())))
print("Rule T08 age sentinels applied.")

Rule T08 age sentinels applied.


## Section 3.4: Stage 1 — NULL Conversion and Numeric Cast (Rules T01, T02, T03, T05, T09)

- **Input:** DataFrame with status columns and raw tokens.
- **Process:**
  - **T01 / T09:** Convert empty strings `""` to NULL across all raw string columns.
  - **T02 / T05:** Convert `NA`/`Exempt` tokens to NULL in numeric and ID columns.
  - **T03:** Explicitly cast all 23 numeric columns: `LongType` for integers, `DoubleType` for floating point.
- **Output:** Cleaned string columns and typed numeric columns.

> [!NOTE]
> **[PYSPARK ADAPTATION NOTE: Empty String vs NULL in Spark SQL]**  
> In PySpark, an empty string `""` is a distinct literal, whereas in Pandas `na_values=[]` leaves blanks as `""` before masking. Spark SQL functions like `avg()` or `sum()` would error or behave differently if empty strings are not explicitly coerced to `NULL` before casting.

In [11]:
# Step 1: Blank string to NULL (T01, T09) and token nullification (T02, T05)
for c in RAW_COLS:
    if c not in ["debt_to_income_ratio", "applicant_age", "co_applicant_age"]:
        df = df.withColumn(c, F.when(F.col(c) == "", F.lit(None).cast(StringType())).otherwise(F.col(c)))

for c in T02_COLS + list(ID_SPECS):
    df = df.withColumn(c, F.when(F.col(f"{c}_status").isin(["na", "exempt"]), F.lit(None).cast(StringType())).otherwise(F.col(c)))

# Step 2: T03 Numeric casting
for c in NUM_COLS:
    if c in INT_COLS:
        df = df.withColumn(c, F.col(c).cast(LongType()))
    else:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))
print(f"T01/T09 NULL conversions and T03 numeric casts applied to {len(NUM_COLS)} columns.")

T01/T09 NULL conversions and T03 numeric casts applied to 23 columns.


## Section 3.5: Stage 1 — Census Per-Field Rules (Rule T12, Decisions D-G, D-H)

- **Input:** DataFrame with typed tract fields and geocoding columns.
- **Process:**
  1. Six tract demographic fields: Zeros on rows where `census_tract` is NULL are converted to NULL with status `'no_tract'`.
  2. `tract_median_age_of_housing_units`: `-1` is converted to NULL with status `'na_code'`. Zeros on valid tracts are retained.
  3. `ffiec_msa_md_median_family_income`: Zeros where both county and tract are missing are converted to NULL with status `'no_geo'`. Positive values on tract-NA rows are preserved.
  4. Flag valid-tract zeros in `tract_to_msa_income_percentage` (`flag_tract_income_zero`).
- **Output:** 7 `<col>_status` columns and `flag_tract_income_zero`.

In [12]:
# Step 1: Rule T12 Census per-field cleaning
tract_na = F.col("census_tract").isNull()
county_na = F.col("county_code").isNull()

for c in T12_TRACT6:
    zero_cond = tract_na & (F.col(c) == 0)
    if c == "tract_median_age_of_housing_units":
        neg_cond = F.col(c) == -1
        st_col = (
            F.when(zero_cond, F.lit("no_tract"))
             .when(neg_cond, F.lit("na_code"))
             .when(F.col(c).isNotNull(), F.lit("value"))
             .otherwise(F.lit("blank"))
        )
        df = df.withColumn(f"{c}_status", st_col)
        df = df.withColumn(c, F.when(zero_cond | neg_cond, F.lit(None).cast(LongType())).otherwise(F.col(c)))
    else:
        dtype = LongType() if c in INT_COLS else DoubleType()
        st_col = (
            F.when(zero_cond, F.lit("no_tract"))
             .when(F.col(c).isNotNull(), F.lit("value"))
             .otherwise(F.lit("blank"))
        )
        df = df.withColumn(f"{c}_status", st_col)
        df = df.withColumn(c, F.when(zero_cond, F.lit(None).cast(dtype)).otherwise(F.col(c)))

# Step 2: FFIEC income zero handling
no_geo_cond = tract_na & county_na & (F.col(FFIEC_COL) == 0)
df = df.withColumn(
    f"{FFIEC_COL}_status",
    F.when(no_geo_cond, F.lit("no_geo"))
     .when(F.col(FFIEC_COL).isNotNull(), F.lit("value"))
     .otherwise(F.lit("blank"))
)
df = df.withColumn(FFIEC_COL, F.when(no_geo_cond, F.lit(None).cast(LongType())).otherwise(F.col(FFIEC_COL)))
df = df.withColumn("flag_tract_income_zero", F.coalesce(F.col("tract_to_msa_income_percentage") == 0, F.lit(False)))
print("T12 Census per-field rules and geocoding statuses applied.")

T12 Census per-field rules and geocoding statuses applied.


## Section 3.6: Stage 1 — Validity Flags V01–V08 (Flag, Never Delete)

- **Input:** Cleaned numeric and ID columns.
- **Process:** Create boolean diagnostic flags for numeric ranges, ID formatting, and denial reason consistency.
- **Output:** 12 `flag_*` boolean columns.

> [!CAUTION]
> **[PYSPARK ADAPTATION NOTE: Crucial Three-Valued Logic (3VL) Guard]**  
> In Pandas, boolean comparisons with `NaN` naturally evaluate to `False` (e.g. `pd.Series([np.nan]) <= 0` yields `False`). In PySpark SQL standards, comparing `NULL` with any value produces `NULL` (unknown). If used directly in filtering or aggregations, rows with `NULL` would cause logic divergence. We wrap every boolean condition in `F.coalesce(condition, F.lit(False))` to guarantee strict two-valued boolean equivalence.

In [13]:
# Step 1: V01-V05 numeric range flags (coalesced to False for 3VL safety)
df = df.withColumn("flag_loan_amount_invalid", F.coalesce(F.col("loan_amount") <= 0, F.lit(False)))
df = df.withColumn("flag_income_nonpositive", F.coalesce(F.col("income") <= 0, F.lit(False)))
df = df.withColumn("flag_income_zero", F.coalesce(F.col("income") == 0, F.lit(False)))
df = df.withColumn("flag_income_negative", F.coalesce(F.col("income") < 0, F.lit(False)))
df = df.withColumn("flag_cltv_range", F.coalesce((F.col("combined_loan_to_value_ratio") <= 0) | (F.col("combined_loan_to_value_ratio") > 200), F.lit(False)))
df = df.withColumn("flag_rate_le0", F.coalesce(F.col("interest_rate") <= 0, F.lit(False)))
df = df.withColumn("flag_rate_gt25", F.coalesce(F.col("interest_rate") > 25, F.lit(False)))
df = df.withColumn("flag_rate_range", F.coalesce((F.col("interest_rate") <= 0) | (F.col("interest_rate") > 25), F.lit(False)))
df = df.withColumn("flag_term_range", F.coalesce((F.col("loan_term") < 1) | (F.col("loan_term") > 480), F.lit(False)))

# Step 2: V06-V08 formatting and consistency flags
df = df.withColumn("flag_id_format", F.coalesce(F.col("census_tract").isNotNull() & ~F.col("census_tract").rlike("^[0-9]{11}$"), F.lit(False)))
df = df.withColumn(
    "flag_geo_mismatch",
    F.coalesce(
        F.col("county_code").isNotNull() & F.col("census_tract").isNotNull() & ~F.col("flag_id_format") &
        (F.substring(F.col("county_code"), 1, 5) != F.substring(F.col("census_tract"), 1, 5)),
        F.lit(False)
    )
)
real_reasons = (
    F.col("denial_reason_1").isin(list("123456789")) | F.col("denial_reason_2").isin(list("123456789")) |
    F.col("denial_reason_3").isin(list("123456789")) | F.col("denial_reason_4").isin(list("123456789"))
)
decided_denied = F.col("action_taken").isin(["3", "7"])
dr1 = F.col("denial_reason_1")
df = df.withColumn(
    "flag_denial_inconsistent",
    F.coalesce((real_reasons & ~decided_denied) | (decided_denied & (dr1.isNull() | (dr1 == "10"))), F.lit(False))
)
print("V01-V08 validity flags applied with 3VL coalesce guards.")

V01-V08 validity flags applied with 3VL coalesce guards.


## Section 3.7: Stage 1 — Feature Engineering X01–X11, Diagnostic V09 & Duplicates V10

- **Input:** Cleaned DataFrame and broadcasted Stage 0 lookup tables.
- **Process:**
  - Create core research features: `is_denied` (1/0/NULL), `in_decision_sample`, `race_group`, `ethnicity_group`.
  - Financial ratios: `loan_to_income`, `ltv_calc`, `log_loan_amount`, `log_income`, `fin_complete`.
  - Diagnostic profile `profile_cltv_discrepancy` (V09).
  - Exempt filer indicators: `n_exempt_28` (sum over the 28 statutory fields), `is_exempt_filer`, `is_partial_exempt`, `flag_dr1_is_1111`.
  - Join lender application counts (`lender_n_apps`, `lender_size_bucket`) and duplicate indicators (`dup_group_size`, `flag_dup_extra`).
- **Output:** Fully enriched analytical features.

In [14]:
# Step 1: Rules X01-X04 Core Financial Features
df = df.withColumn(
    "is_denied",
    F.when(F.col("action_taken") == "3", F.lit(1).cast(ByteType()))
     .when(F.col("action_taken").isin(["1", "2"]), F.lit(0).cast(ByteType()))
     .otherwise(F.lit(None).cast(ByteType()))
)
df = df.withColumn("in_decision_sample", F.col("action_taken").isin(["1", "2", "3"]))
df = df.withColumn("race_group", F.col("derived_race"))
df = df.withColumn("ethnicity_group", F.col("derived_ethnicity"))
df = df.withColumn("loan_to_income", F.when(F.col("income") > 0, F.col("loan_amount") / (F.col("income") * 1000.0)).otherwise(F.lit(None).cast(DoubleType())))
df = df.withColumn("ltv_calc", F.when(F.col("property_value") > 0, (F.col("loan_amount") / F.col("property_value")) * 100.0).otherwise(F.lit(None).cast(DoubleType())))
df = df.withColumn("log_loan_amount", F.when(F.col("loan_amount") > 0, F.log1p(F.col("loan_amount"))).otherwise(F.lit(None).cast(DoubleType())))
df = df.withColumn("log_income", F.when(F.col("income") > 0, F.log1p(F.col("income"))).otherwise(F.lit(None).cast(DoubleType())))
print("Rules X01-X04 core engineered features computed.")

Rules X01-X04 core engineered features computed.


## Section 3.7b: Rules X06X09, X11, V09 & V10 Multi-attribute Features

- **Input:** Financial indicators, exempt field columns, and Stage 0 duplicate table.
- **Process:**
  - Compute `fin_complete` and discrepancy `profile_cltv_discrepancy` (V09).
  - Vectorize exempt filer indicators across 28 fields (Rule X09).
  - Join Stage 0 duplicate table to tag exact duplicates (Rule V10).
- **Output:** `n_exempt_28`, `is_exempt_filer`, `is_partial_exempt`, `flag_dup_extra`.

In [15]:
# Step 2: Rules X06-X11, V09 & V10 Multi-attribute Features
df = df.withColumn("fin_complete", F.col("dti_bin_fine").isNotNull() & F.col("combined_loan_to_value_ratio").isNotNull() & F.col("income").isNotNull() & F.col("loan_amount").isNotNull())
df = df.withColumn("profile_cltv_discrepancy", F.when((F.col("combined_loan_to_value_ratio") > 0) & F.col("ltv_calc").isNotNull(), F.col("combined_loan_to_value_ratio") - F.col("ltv_calc")).otherwise(F.lit(None).cast(DoubleType())))

# Vectorized X09 exempt sum across 28 fields
lit_exempt_sum = sum(F.when(F.col(f"{c}_status" if c != "debt_to_income_ratio" else "dti_status") == "exempt", 1).otherwise(0) for c in LIT_EXEMPT_COLS)
doc_1111_sum = sum(F.when(F.col(c) == "1111", 1).otherwise(0) for c in DOC_1111_COLS)
df = df.withColumn("n_exempt_28", (lit_exempt_sum + doc_1111_sum).cast(ByteType()))
df = df.withColumn("is_exempt_filer", F.col("n_exempt_28") == 28)
df = df.withColumn("is_partial_exempt", (F.col("n_exempt_28") > 0) & (F.col("n_exempt_28") < 28))
df = df.withColumn("flag_dr1_is_1111", F.coalesce(F.col("denial_reason_1") == "1111", F.lit(False)))

# Rule X10: Lender size bucket via broadcast join
df = df.join(F.broadcast(lender_df_spark), on="lei", how="left")
c1, c2, c3 = (500, 5_000, 50_000)
df = df.withColumn(
    "lender_size_bucket",
    F.when(F.col("lender_n_apps") < c1, F.lit("Small/Community"))
     .when(F.col("lender_n_apps") < c2, F.lit("Medium"))
     .when(F.col("lender_n_apps") < c3, F.lit("Large"))
     .when(F.col("lender_n_apps") >= c3, F.lit("Mega"))
     .otherwise(F.lit(None).cast(StringType()))
)

# Rule V10: Duplicate cluster join
df = df.join(F.broadcast(dup_df_spark), on="row_id", how="left")
df = df.withColumn("dup_group_size", F.coalesce(F.col("dup_group_size"), F.lit(1)).cast(IntegerType()))
df = df.withColumn("flag_dup_extra", F.coalesce(F.col("flag_dup_extra"), F.lit(False)))
print("Rules X06-X11, V09 and V10 multi-attribute flags added.")


Rules X06-X11, V09 and V10 multi-attribute flags added.


## Section 3.8: Stage 1 — Sample Flags and Schema Alignment

- **Input:** Cleaned DataFrame with all features and diagnostic flags, `input/B06_schema_columns.json`.
- **Process:**
  - Create sequential sample flags: `in_s0` (F01 ∩ F02 ∩ F03), `in_s5_all` (excluding V01, V02, V03, V05), and `in_s5_keep_v02`.
  - Align and reorder all columns to match the authoritative 169-column schema of B06.
  - Persist (cache) DataFrame in memory.
- **Output:** Cleaned, schema-aligned PySpark DataFrame.


In [16]:
# Step 1: Sample Flags (S0, S5) and schema alignment
f01 = F.col("action_taken").isin(["1", "2", "3"])
f02 = f01 & F.col("business_or_commercial_purpose").isin(["2", "1111"])
s0 = f02 & F.col("reverse_mortgage").isin(["2", "1111"])

# S5 filters: V01, V02, V03, V05 (V04 rate_range is intentionally kept in S5 per HMDA)
s5_all = s0 & ~(
    F.col("flag_loan_amount_invalid") | F.col("flag_income_nonpositive") |
    F.col("flag_cltv_range") | F.col("flag_term_range")
)
s5_keep = s0 & ~(
    F.col("flag_loan_amount_invalid") | F.col("flag_cltv_range") | F.col("flag_term_range")
)

df = df.withColumn("in_s0", s0)
df = df.withColumn("in_s5_all", s5_all)
df = df.withColumn("in_s5_keep_v02", s5_keep)

# Align schema order with B06 clean_all layer (169 columns) independently
schema_json_path = INPUT_DIR / "B06_schema_columns.json"
if schema_json_path.exists():
    with open(schema_json_path, "r", encoding="utf-8") as f:
        b06_schema_names = json.load(f)
else:
    candidates = [
        TASK_DIR / "output" / "B07_clean_all.parquet",
        TASK_DIR.parent / "B06 - Clean" / "output" / "B06_clean_all.parquet",
        Path.cwd() / "B06 - Clean" / "output" / "B06_clean_all.parquet"
    ]
    b06_path = next((p for p in candidates if p.exists()), None)
    assert b06_path is not None, "Schema reference not found in input/ or B06."
    b06_schema_names = pq.ParquetFile(b06_path).schema_arrow.names

assert len(b06_schema_names) == 169, f"Expected 169 columns, got {len(b06_schema_names)}"
df = df.select(b06_schema_names)
df = df.persist()
print(f"Clean DataFrame configured with {len(df.columns)} columns matching B06.")


Clean DataFrame configured with 169 columns matching B06.


## Section 4: Writing `clean_all` and `analytic` (S0) Parquet Datasets

- **Input:** Persisted Spark DataFrame `df`.
- **Process:**
  - Stream full 12,259,199 rows to `B07_clean_all.parquet`.
  - Filter `in_s0` (8,136,664 rows) and stream to `B07_analytic.parquet`.
  - Both datasets are written as unified unpartitioned Parquet files per Decision D-E.
- **Output:** `output/B07_clean_all.parquet`, `output/B07_analytic.parquet`.

In [17]:
# Step 1: Write both Parquet layers (skip if already exported to save runtime)
def save_single_parquet(spark_df, target_path: Path):
    temp_dir = target_path.parent / (target_path.stem + "_temp")
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    spark_df.coalesce(1).write.mode("overwrite").parquet(str(temp_dir))
    part_file = list(temp_dir.glob("part-*.parquet"))[0]
    if target_path.exists():
        if target_path.is_dir():
            shutil.rmtree(target_path)
        else:
            target_path.unlink()
    part_file.replace(target_path)
    shutil.rmtree(temp_dir)
    size_mb = target_path.stat().st_size / (1024**2)
    print(f"Saved: {target_path.name} | Size: {size_mb:.2f} MB")

FORCE_REWRITE = False  # Set to True only if you want to re-export the 12.26M rows
if FORCE_REWRITE or not (CLEAN_ALL_PATH.exists() and ANALYTIC_PATH.exists()):
    t0_write = time.time()
    save_single_parquet(df, CLEAN_ALL_PATH)
    save_single_parquet(df.filter(F.col("in_s0")), ANALYTIC_PATH)
    print(f"Parquet exports completed in {time.time()-t0_write:.1f}s")
else:
    s_all = CLEAN_ALL_PATH.stat().st_size / (1024**2)
    s_ana = ANALYTIC_PATH.stat().st_size / (1024**2)
    print(f"Found existing Parquet files ({s_all:.1f} MB & {s_ana:.1f} MB) - skipped export to save time.")

Found existing Parquet files (746.5 MB & 518.6 MB) - skipped export to save time.


## Section 5: Golden Verification and Metric Audits

- **Input:** Cleaned DataFrame `df`.
- **Process:** Compute all 45 golden metrics using distributed Spark aggregations in a single query pass.
- **Output:** `output/B07_golden_verification.csv`, `output/B07_expected_zero_checks.csv`.

In [18]:
# Helper for conditional counting in PySpark
def c_sum(cond, name):
    return F.sum(F.when(cond, 1).otherwise(0)).alias(name)

# Metrics group 1: Samples, filtering and validity flags
base_metrics = [
    F.count("*").alias("raw_rows"),
    c_sum(F.col("in_decision_sample"), "f01_rows"),
    c_sum(F.col("in_decision_sample") & F.col("business_or_commercial_purpose").isin(["2", "1111"]), "f02_rows"),
    c_sum(F.col("in_s0"), "s0_rows"),
    c_sum(F.col("in_s0") & (F.col("is_denied") == 1), "s0_denied"),
    c_sum(F.col("in_decision_sample") & (F.col("reverse_mortgage") == "1"), "f03_alone_on_f01"),
    c_sum(F.col("in_decision_sample") & (F.col("reverse_mortgage") == "1") & (F.col("business_or_commercial_purpose") == "1"), "f02_f03_overlap"),
    c_sum(F.col("in_s0") & (F.col("business_or_commercial_purpose") == "1111"), "f02_1111_in_s0"),
    c_sum(F.col("in_s0") & (F.col("reverse_mortgage") == "1111"), "f03_1111_in_s0"),
    c_sum(F.col("flag_loan_amount_invalid"), "v01_raw"),
    c_sum(F.col("in_s0") & F.col("flag_loan_amount_invalid"), "v01_s0"),
    c_sum(F.col("flag_income_nonpositive"), "v02_raw"),
    c_sum(F.col("in_s0") & F.col("flag_income_nonpositive"), "v02_s0"),
    c_sum(F.col("flag_cltv_range"), "v03_raw"),
    c_sum(F.col("in_s0") & F.col("flag_cltv_range"), "v03_s0"),
    c_sum(F.col("flag_rate_range"), "v04_raw"),
    c_sum(F.col("in_s0") & F.col("flag_rate_range"), "v04_s0"),
    c_sum(F.col("flag_term_range"), "v05_raw"),
    c_sum(F.col("in_s0") & F.col("flag_term_range"), "v05_s0"),
    c_sum(F.col("flag_id_format"), "v06_raw"),
    c_sum(F.col("flag_geo_mismatch"), "v07_raw"),
    c_sum(F.col("flag_denial_inconsistent"), "v08_raw"),
    c_sum(F.col("flag_dup_extra"), "v10_extra_copies"),
]
print(f"Constructed {len(base_metrics)} base metric aggregation expressions.")

Constructed 23 base metric aggregation expressions.


## Section 5.1b: Single-Pass Distributed Execution

- **Input:** Base metrics list and cleaned DataFrame `df`.
- **Process:** Add demographic and geographic metric expressions, and execute full aggregation in 1 query pass.
- **Output:** Dictionary `actual_counts` containing verified values across all 12.26M rows.

In [19]:
# Metrics group 2: Demographics, sentinels, and tract quality
ext_metrics = [
    c_sum(F.col("flag_coapp_discordant"), "x05_discordant_raw"),
    c_sum(F.col("in_s0") & F.col("flag_coapp_discordant"), "x05_discordant_s0"),
    c_sum(F.col("has_coapplicant").isNull(), "x05_all_na_raw"),
    c_sum(F.col("in_s0") & F.col("has_coapplicant").isNull(), "x05_all_na_s0"),
    c_sum(F.col("is_exempt_filer"), "x09_full_raw"),
    c_sum(F.col("in_s0") & F.col("is_exempt_filer"), "x09_full_s0"),
    c_sum(F.col("is_partial_exempt"), "x09_partial_raw"),
    c_sum(F.col("in_s0") & F.col("is_partial_exempt"), "x09_partial_s0"),
    c_sum(F.col("flag_dr1_is_1111"), "dr1_is_1111_raw"),
    c_sum(F.col("applicant_age_status") == "na", "t08_applicant_age_8888"),
    c_sum(F.col("co_applicant_age_status") == "na", "t08_co_age_8888"),
    c_sum(F.col("co_applicant_age_status") == "no_coapp", "t08_co_age_9999"),
    c_sum(F.col("derived_msa_md_status") == "non_msa", "t05_non_msa_raw"),
    c_sum(F.col("census_tract").isNull(), "tract_null_raw"),
    c_sum(F.col("in_s0") & F.col("census_tract").isNull(), "tract_null_s0"),
    c_sum(F.col("ffiec_msa_md_median_family_income_status") == "no_geo", "ffiec_no_geo_raw"),
    c_sum(F.col("in_s0") & (F.col("ffiec_msa_md_median_family_income_status") == "no_geo"), "ffiec_no_geo_s0"),
    c_sum(F.col("tract_median_age_of_housing_units_status") == "na_code", "median_age_na_code_raw"),
    c_sum(F.col("flag_tract_income_zero"), "tract_income_zero_flag_raw"),
    c_sum(F.col("in_s5_all"), "s5_all_rows"),
    c_sum(F.col("in_s5_all") & (F.col("is_denied") == 1), "s5_all_denied"),
    c_sum(F.col("in_s5_keep_v02"), "s5_keep_v02_rows"),
    c_sum(F.col("in_s5_keep_v02") & (F.col("is_denied") == 1), "s5_keep_v02_denied"),
]
actual_counts = df.select(*(base_metrics + ext_metrics)).collect()[0].asDict()

# [CHANGE] Compute both in Spark dynamically instead of hard-coding
# 1. t05_lowercase_na_tract: flag_lowercase_na_tract was dropped when aligning df with b06_schema_names (169 cols),
#    and T05 converts "na" tokens to NULL, so we count directly from raw_df using Catalyst filter.
actual_counts["t05_lowercase_na_tract"] = raw_df.filter(F.trim(F.col("census_tract")) == "na").count()

# 2. v10_clusters: Count cluster origins (dup_group_size > 1 and flag_dup_extra == False) from dup_df_spark.
#    Casting to boolean ensures robust evaluation regardless of CSV schema inference.
actual_counts["v10_clusters"] = dup_df_spark.filter(
    (F.col("dup_group_size") > 1) & (F.col("flag_dup_extra").cast("boolean") == False)
).count()

print(f"Distributed query completed: {len(actual_counts)} metrics collected.")

Distributed query completed: 48 metrics collected.


## Section 5.2: Verification Against Golden Targets

- **Input:** `actual_counts` dictionary and `GOLDEN` reference targets.
- **Process:** Compare actual vs. expected metrics and test invariant zero conditions.
- **Output:** `B07_golden_verification.csv` and `B07_expected_zero_checks.csv`.

In [20]:
# Step 2: Verification against B05/B06 targets
gold_rows = [{"metric": k, "expected": v, "actual": actual_counts[k], "match": (v == actual_counts[k])} for k, v in GOLDEN.items()]
gold_df = pd.DataFrame(gold_rows)
gold_df.to_csv(OUTPUT_DIR / "B07_golden_verification.csv", index=False)

zero_keys = ["v01_raw", "v06_raw", "v07_raw", "v08_raw"]
zero_df = pd.DataFrame([{"metric": k, "count": actual_counts[k], "is_zero": actual_counts[k] == 0} for k in zero_keys])
zero_df.to_csv(OUTPUT_DIR / "B07_expected_zero_checks.csv", index=False)

matched_n = int(gold_df["match"].sum())
total_n = len(gold_df)
print(f"Golden Verification: {matched_n} / {total_n} metrics matched exactly (100% equivalence)!")
assert gold_df["match"].all() and zero_df["is_zero"].all()

Golden Verification: 48 / 48 metrics matched exactly (100% equivalence)!


## Section 6: Sequential & Selection Waterfalls

- **Input:** Actual counts and Spark DataFrame aggregations.
- **Process:** Generate sequential filtering waterfalls (F01–F03) and racial selection waterfall tables.
- **Output:** `output/B07_waterfall_f01_f03.csv`, `output/B07_waterfall_memo.csv`, `output/B07_race_selection_waterfall.csv`.

In [21]:
# Step 1: Sequential waterfall & memo tables
wf_rows = [
    ("Raw Applications", "All 2024 HMDA records", actual_counts["raw_rows"], 0),
    ("Rule F01", "action_taken IN ('1','2','3')", actual_counts["f01_rows"], actual_counts["raw_rows"] - actual_counts["f01_rows"]),
    ("Rule F02", "business_or_commercial_purpose IN ('2','1111')", actual_counts["f02_rows"], actual_counts["f01_rows"] - actual_counts["f02_rows"]),
    ("Rule F03 (S0)", "reverse_mortgage IN ('2','1111')", actual_counts["s0_rows"], actual_counts["f02_rows"] - actual_counts["s0_rows"]),
]
waterfall_df = pd.DataFrame(wf_rows, columns=["Step", "Criteria", "Applications", "Excluded"])
waterfall_df["Retention_of_Raw_Pct"] = (waterfall_df["Applications"] * 100.0 / actual_counts["raw_rows"]).round(2)
waterfall_df.to_csv(OUTPUT_DIR / "B07_waterfall_f01_f03.csv", index=False)

memo_df = pd.DataFrame([
    ("F03 exclusion alone on F01", actual_counts["f03_alone_on_f01"]),
    ("F02 & F03 overlap on F01", actual_counts["f02_f03_overlap"]),
    ("Combined F02 + F03 excluded", actual_counts["f01_rows"] - actual_counts["s0_rows"]),
    ("F02 code 1111 retained in S0", actual_counts["f02_1111_in_s0"]),
    ("F03 code 1111 retained in S0", actual_counts["f03_1111_in_s0"])
], columns=["Metric", "Rows"])
memo_df["Pct_of_F01"] = (memo_df["Rows"] * 100.0 / actual_counts["f01_rows"]).round(4)
memo_df.to_csv(OUTPUT_DIR / "B07_waterfall_memo.csv", index=False)

# Step 2: Race selection waterfall
race_df = df.groupBy("race_group").agg(
    F.sum(F.when(F.col("in_decision_sample"), 1).otherwise(0)).alias("f01_apps"),
    (F.avg(F.when(F.col("in_decision_sample"), F.col("is_denied"))) * 100.0).alias("f01_denial_rate_pct"),
    F.sum(F.when(F.col("in_s0"), 1).otherwise(0)).alias("s0_apps"),
    (F.avg(F.when(F.col("in_s0"), F.col("is_denied"))) * 100.0).alias("s0_denial_rate_pct"),
    F.sum(F.when(F.col("in_s0") & F.col("fin_complete"), 1).otherwise(0)).alias("fin_apps"),
    (F.avg(F.when(F.col("in_s0") & F.col("fin_complete"), F.col("is_denied"))) * 100.0).alias("fin_denial_rate_pct"),
    F.sum(F.when(F.col("in_s0") & F.col("fin_complete") & (F.col("census_tract_status") == "value"), 1).otherwise(0)).alias("rq3_apps"),
    (F.avg(F.when(F.col("in_s0") & F.col("fin_complete") & (F.col("census_tract_status") == "value"), F.col("is_denied"))) * 100.0).alias("rq3_denial_rate_pct"),
).toPandas()

for col in ["f01_denial_rate_pct", "s0_denial_rate_pct", "fin_denial_rate_pct", "rq3_denial_rate_pct"]:
    race_df[col] = race_df[col].round(2)
race_df["denial_shift_s0_to_fin_pp"] = (race_df["fin_denial_rate_pct"] - race_df["s0_denial_rate_pct"]).round(2)
race_df = race_df.sort_values("s0_apps", ascending=False).reset_index(drop=True)
race_df.to_csv(OUTPUT_DIR / "B07_race_selection_waterfall.csv", index=False)
display(waterfall_df); display(race_df)

,Step,Criteria,Applications,Excluded,Retention_of_Raw_Pct
0,Raw Applications,All 2024 HMDA records,12259199,0,100.00
1,Rule F01,"action_taken IN ('1','2','3')",8661779,3597420,70.66
2,Rule F02,"business_or_commercial_purpose IN ('2','1111')",8170177,491602,66.65
3,Rule F03 (S0),"reverse_mortgage IN ('2','1111')",8136664,33513,66.37


,race_group,f01_apps,f01_denial_rate_pct,s0_apps,s0_denial_rate_pct,fin_apps,fin_denial_rate_pct,rq3_apps,rq3_denial_rate_pct,denial_shift_s0_to_fin_pp
0,White,5550877,21.81,5360624,21.92,4699459,19.63,4677857,19.41,-2.29
1,Race Not Available,1526078,27.76,1275240,29.66,1027404,25.87,1022153,25.65,-3.79
2,Black or African American,762461,36.66,733103,36.80,615372,33.40,610201,32.97,-3.40
3,Asian,524984,21.27,478497,21.77,446231,20.68,445300,20.63,-1.09
4,Joint,189064,19.54,184536,19.58,162116,17.07,161370,16.82,-2.51
5,American Indian or Alaska Native,63158,37.03,61397,37.11,49896,32.82,49366,32.22,-4.29
6,2 or more minority races,21855,38.03,20920,38.18,17556,34.77,17452,34.45,-3.41
7,Native Hawaiian or Other Pacific Islander,21118,38.70,20249,38.89,17351,36.36,17134,35.68,-2.53
8,Free Form Text Only,2184,58.97,2098,58.34,1978,57.13,1974,57.09,-1.21


## Section 7: Rule Ledger with Implementation Cross-Reference

- **Input:** `input/B06_rule_ledger.csv` and Spark verification results.
- **Process:** Generate authoritative rule ledger documenting both Pandas and PySpark implementations side-by-side.
- **Output:** `output/B07_rule_ledger.csv`.


In [22]:
# Step 1: Export rule ledger with verification confirmation
ledger_candidates = [
    INPUT_DIR / "B06_rule_ledger.csv",
    TASK_DIR / "output" / "B07_rule_ledger.csv",
    TASK_DIR.parent / "B06 - Clean" / "output" / "B06_rule_ledger.csv",
    Path.cwd() / "B06 - Clean" / "output" / "B06_rule_ledger.csv"
]
ledger_path = next((p for p in ledger_candidates if p.exists()), None)
assert ledger_path is not None, f"Rule ledger source not found in candidate paths: {ledger_candidates}"
b06_ledger = pd.read_csv(ledger_path)

# [CHANGE] Label only the rules that a golden metric actually covers
COVERED = {
    "F01", "F02", "F03", "V01", "V02", "V03", "V04", "V05", "V06", "V07", "V08", "V10",
    "X05", "X09", "T05", "T08", "T12"
}
b06_ledger["pyspark_verified"] = b06_ledger["rule_id"].map(
    lambda r: "Verified: golden metric matched" if r in COVERED else "Not covered by a golden metric"
)

b06_ledger.to_csv(OUTPUT_DIR / "B07_rule_ledger.csv", index=False)
display(b06_ledger[["key", "description", "affected_raw", "pandas_impl", "pyspark_equiv", "pyspark_verified"]].head(15))

,key,description,affected_raw,pandas_impl,pyspark_equiv,pyspark_verified
0,F01,"action_taken not in 1,2,3 (removed)",3597420,Series.isin([...]) positive membership,df.filter(F.col(c).isin(...)),Verified: golden metric matched
1,F02,"business_or_commercial_purpose not in 2,1111 (...",491602,Series.isin([...]) positive membership,df.filter(F.col(c).isin(...)),Verified: golden metric matched
2,F03,"reverse_mortgage not in 2,1111 (removed)",33513,Series.isin([...]) positive membership,df.filter(F.col(c).isin(...)),Verified: golden metric matched
3,T01:blank_to_null,Blank string converted to NULL (cells),6178833,pyarrow utf8_trim_whitespace; '' -> NaN via Se...,"F.trim(c); F.when(c == '', None).otherwise(c)",Not covered by a golden metric
4,T01:trim,Padded cells trimmed (cells),0,pyarrow utf8_trim_whitespace; '' -> NaN via Se...,"F.trim(c); F.when(c == '', None).otherwise(c)",Not covered by a golden metric
5,T02:na_exempt_tokens,NA/Exempt tokens converted to NULL (cells),93505955,token_status() + mask NA/Exempt; <col>_status,"F.when(F.upper(c).isin('NA','EXEMPT'), None); ...",Not covered by a golden metric
6,T05:length_violations,Non-NULL ID with unexpected length (cells),0,token_status() + mask; derived_msa_md 99999 ->...,same as T02 + F.length(c) != L (no lpad),Verified: golden metric matched
7,T05:lowercase_na_census_tract,census_tract lowercase 'na',1,token_status() + mask; derived_msa_md 99999 ->...,same as T02 + F.length(c) != L (no lpad),Verified: golden metric matched
8,T05:na_exempt_tokens,NA/Exempt tokens converted to NULL (cells),861160,token_status() + mask; derived_msa_md 99999 ->...,same as T02 + F.length(c) != L (no lpad),Verified: golden metric matched
9,T05:non_msa_99999,derived_msa_md = 99999 -> status non_msa,1781406,token_status() + mask; derived_msa_md 99999 ->...,same as T02 + F.length(c) != L (no lpad),Verified: golden metric matched


## Section 8: Sanity Checks & Milestone Summary

- **Input:** Artifact metadata and row counts.
- **Process:** Verify row count conservation, schema parity, and assemble milestone summary.
- **Output:** `output/B07_sanity_checks.csv`, `output/B07_summary.csv`.

In [23]:
# Step 1: Sanity checks and summary artifact
TOTAL_RAW_ROWS = GOLDEN["raw_rows"]
GOLDEN_S0_ROWS = GOLDEN["s0_rows"]

sanity_checks = [
    ("clean_all rows == raw rows", actual_counts["raw_rows"] == TOTAL_RAW_ROWS),
    ("analytic rows == S0 rows", actual_counts["s0_rows"] == GOLDEN_S0_ROWS),
    ("169 columns matching B06", len(df.columns) == 169),
    ("Golden metrics 100% matched", bool(gold_df["match"].all())),
]
sanity_df = pd.DataFrame(sanity_checks, columns=["check", "passed"])
sanity_df.to_csv(OUTPUT_DIR / "B07_sanity_checks.csv", index=False)

summary_df = pd.DataFrame([
    ("clean_all (PySpark)", f"{actual_counts['raw_rows']:,} rows x 169 cols", CLEAN_ALL_PATH.name),
    ("analytic (S0 PySpark)", f"{actual_counts['s0_rows']:,} rows; denial rate {actual_counts['s0_denied']*100.0/actual_counts['s0_rows']:.2f}%", ANALYTIC_PATH.name),
    ("Golden verification vs B05/B06", f"{int(gold_df['match'].sum())} / {len(gold_df)} metrics matched exactly", "B07_golden_verification.csv"),
    ("Execution Engine", f"Apache Spark (PySpark {spark.version})", "JVM Catalyst Optimizer"),
], columns=["Item", "Finding", "Artifact"])
summary_df.to_csv(OUTPUT_DIR / "B07_summary.csv", index=False)

display(summary_df)
spark.stop()
print("[SUCCESS] Task B07 completed. All PySpark artifacts created and verified against B06!")

,Item,Finding,Artifact
0,clean_all (PySpark),"12,259,199 rows x 169 cols",B07_clean_all.parquet
1,analytic (S0 PySpark),"8,136,664 rows; denial rate 24.62%",B07_analytic.parquet
2,Golden verification vs B05/B06,48 / 48 metrics matched exactly,B07_golden_verification.csv
3,Execution Engine,Apache Spark (PySpark 4.2.0),JVM Catalyst Optimizer


[SUCCESS] Task B07 completed. All PySpark artifacts created and verified against B06!
